In [41]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from tqdm import tqdm
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

In [42]:
df_final = pd.read_excel("data/processed/df_final.xlsx")
df_final = df_final.sort_values("Date").reset_index(drop=True) 

In [43]:
covariates = ["dolvol_lag2", "maxret", "retvol", "mom36m", "mom12m", "mom6m", "mom1m", "chmom", "turn", "indmom", "baspread", "illiq", "stdturn", "beta", "beta_squared",
"idiovol", "mvel1", "agr", "cashpr", "chinv", "chsh", "depr", "dy", "ep", "invest", "rd_mve", "sp", "nincr"]

I. Fonctions : normalisation et gestion des NaN

In [ ]:
def generate_time_splits(df, date_col='Date',
                                   val_months=12,
                                   test_months=12,
                                   step_months=12,
                                   min_train_months=306):
    """
    Génère des splits temporels:
    - Train cumulatif (augmente d'un an à chaque refit)
    - Train 306 mois (= 85% de la data)
    - Validation = fenêtre fixe glissante de 1 an
    - Test = 1 an 
    - Avance de step_months à chaque itération : 1 an

    Paramètres :
    - df : DataFrame trié par date
    - date_col : nom de la colonne des dates
    - val_months : taille de la validation
    - test_months : taille du test 
    - step_months : pas de glissement
    - min_train_months : nombre minimum de mois de train initial

    Retour :
    - splits : liste de tuples (train_idx, val_idx, test_idx)
    """

    #On coupe chronologiquement donc on trie par date 
    df = df.sort_values(date_col).reset_index(drop=True)
    dates = sorted(df[date_col].unique())
    total_months = len(dates)

    splits = []

    #On démarre après avoir au moins min_train_months pour le train
    start = min_train_months
    while True:
        train_end = start  # train va de 0 jusqu'à train_end
        val_start = train_end
        val_end = val_start + val_months
        test_start = val_end
        test_end = test_start + test_months

        #Stop quand on a plus assez pour test
        if test_end > total_months:
            break

        train_dates = dates[:train_end]  
        val_dates = dates[val_start:val_end]
        test_dates = dates[test_start:test_end]

        train_idx = df[df[date_col].isin(train_dates)].index.tolist()
        val_idx = df[df[date_col].isin(val_dates)].index.tolist()
        test_idx = df[df[date_col].isin(test_dates)].index.tolist()

        splits.append((train_idx, val_idx, test_idx))

        #Décale fenêtre de un → on réactualise tous les 1 ans
        start += step_months

    return splits

In [45]:
def preprocess_split(X_train, X_val, X_test, covariates):
    """
    Impute les NaN par moyenne par Ticker (fit sur train),
    puis normalise chaque covariable entre -1 et 1 par date (rang cross-sectionnel).

    Paramètres
    ----------
    X_train, X_val, X_test : DataFrames bruts (avec 'Ticker' et 'Date')
    covariates : liste des colonnes numériques à traiter

    Retour
    ------
    X_train_scaled, X_val_scaled, X_test_scaled : DataFrames transformés (covariates seulement)
    """
    #Gestion des NaN : moyenne par Ticker calculée sur le train
    means_by_ticker = x_train.groupby("Ticker")[covariates].mean(numeric_only=True)

    def fill_na_with_means(df):
        df = df.copy()
        for col in covariates:
            #On remplace NaN par la moyenne du ticker
            df[col] = df.apply(
                lambda row: means_by_ticker[col][row["Ticker"]] 
                            if pd.isna(row[col]) and row["Ticker"] in means_by_ticker.index 
                            else row[col],
                axis=1
            )
        return df

    x_train_imp = fill_na_with_means(x_train)
    x_val_imp   = fill_na_with_means(x_val)
    x_test_imp  = fill_na_with_means(x_test)

    #Normalisation cross-sectionnelle : par date
    def normalize_by_date(df):
        df = df.copy()
        out = pd.DataFrame(index=df.index, columns=covariates)
        for date_key, group_idx in df.groupby("Date").groups.items():
            sub = df.loc[group_idx, covariates]
            for cov in covariates:
                temp = sub[cov].dropna().sort_values()
                n = len(temp)
                if n == 1:
                    scores = pd.Series([0.0], index=temp.index)
                else:
                    scores = pd.Series(
                        2 * np.arange(n) / (n - 1) - 1, index=temp.index
                    )
                out.loc[temp.index, cov] = scores
        return out.astype(float)

    x_train_scaled = normalize_by_date(x_train_imp)
    x_val_scaled   = normalize_by_date(x_val_imp)
    x_test_scaled  = normalize_by_date(x_test_imp)

    #garde l'information de la date et du ticker
    x_train_scaled = pd.concat([x_train_imp[['Ticker','Date']].reset_index(drop=True), x_train_scaled.reset_index(drop=True)], axis=1)
    x_val_scaled   = pd.concat([x_val_imp[['Ticker','Date']].reset_index(drop=True), x_val_scaled.reset_index(drop=True)], axis=1)
    x_test_scaled  = pd.concat([x_test_imp[['Ticker','Date']].reset_index(drop=True), x_test_scaled.reset_index(drop=True)], axis=1)

    return x_train_scaled, x_val_scaled, x_test_scaled

ICI AJOUTER UN CODE POUR VOIR SI y A ENCORE DES NAN ?

In [ ]:
#R²
#Mesures : → peut être à tej 
def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum(y_true**2) 
    return 1 - ss_res/ss_tot if ss_tot != 0 else np.nan

#% ratio:
def success_ratio(y_true, y_pred, ignore_zero=True):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    sign_true = np.sign(y_true)
    sign_pred = np.sign(y_pred)

    if ignore_zero:
        mask = sign_true != 0
        sign_true = sign_true[mask]
        sign_pred = sign_pred[mask]

    if len(sign_true) == 0:
        return np.nan  
    return (sign_true == sign_pred).mean()

#R2 benchmark 
def r2_vs_benchmark(y_true, y_pred_model, y_pred_bench):
    T = len(y_true)
    mspe_model = (1/T) * np.sum((y_true - y_pred_model)**2)
    mspe_bench = (1/T) * np.sum((y_true - y_pred_bench)**2)
    return 1 - (mspe_model / mspe_bench)

In [47]:
#Permet de récupérer x et y 
def get_x_y(df, idx, target="excess_return"):
    subset = df.loc[idx].copy()
    x = subset.drop(columns=[target]) #garde toutes les colonnes mais enlève excess return
    y = subset[target]
    return x, y

In [48]:
#On découpe les splits puis on applique la gestion des NaN et la normalisation définie plus haut
splits = generate_time_splits(df_final)

preprocessed_splits = []

for train_idx, val_idx, test_idx in tqdm(splits):
    x_train, y_train = get_x_y(df_final, train_idx)
    x_val, y_val = get_x_y(df_final, val_idx)
    x_test, y_test = get_x_y(df_final, test_idx)

    #Imputation + Normalisation
    x_train, x_val, x_test = preprocess_split(x_train, x_val, x_test, covariates) #on enlève ticker et date

    preprocessed_splits.append((x_train, y_train, x_val, y_val, x_test, y_test))

100%|██████████| 3/3 [00:28<00:00,  9.61s/it]


BENCHMARK

In [ ]:
"""HISTORICAL AVERAGE

Détail des listes : 

On stocke une seule fois (on split toujours de la même façon, donc c'est commun à tous les modèles):
y_true : les vraies prédictions out-of-sample 
y_trainval_true : les vraies prédictions in-sample
dates_in : les dates pour chaque observation in-sample (utile si on veut les R² in sample)
dates_oos: dates pour chaque observations oos (utile pour récupérer les )
tickers_in : idem
tickers_oos : idem

Pour chaque modèle on stock :
- les prédictions oos : y_pred_model (ici ha)
- les prédictions in-sample : y_trainval_pred_model
- success ratio in : success_ration_in_model
- sucess ratio oos : success_ratio_oos_model
"""

#On stock : y réel et trainval réel pour calculer les r², on garde aussi les dates et tickers pour construire les portefeuilles (partie 3 results) 
y_true = []
y_trainval_true = []
dates_in = []
dates_oos = []
tickers_in = []
tickers_oos = []

y_pred_ha = []
y_trainval_pred_ha = []

r2_in_ha = []
r2_oos_ha = []

success_ratio_in_ha = []
success_ratio_oos_ha = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(preprocessed_splits, start=1):

    # In-sample
    tickers_trainval = pd.concat([x_train['Ticker'], x_val['Ticker']], ignore_index=True)
    dates_trainval = pd.concat([x_train['Date'], x_val['Date']], ignore_index=True)
    y_trainval_all = pd.concat([y_train, y_val], ignore_index=True)
    trainval = pd.DataFrame({'Ticker': tickers_trainval, 'y': y_trainval_all})

    #moyenne historique
    mean_by_ticker = trainval.groupby('Ticker')['y'].mean()
    preds_trainval = trainval['Ticker'].map(mean_by_ticker).values

    y_trainval_true.extend(trainval['y'])
    y_trainval_pred_ha.extend(preds_trainval)
    dates_in.extend(dates_trainval)
    tickers_in.extend(tickers_trainval)

    r2_in = r2(trainval['y'], preds_trainval)
    sr_in = success_ratio(trainval['y'], preds_trainval)
    r2_in_ha.append(r2_in)
    success_ratio_in_ha.append(sr_in)

    # Out-of-sample
    preds_split = [mean_by_ticker.get(tkr, np.nan) for tkr in x_test['Ticker']]
    preds_split = np.array(preds_split)
    r2_out = r2(y_test, preds_split)
    sr_out = success_ratio(y_test, preds_split)

    r2_oos_ha.append(r2_out)
    success_ratio_oos_ha.append(sr_out)

    y_pred_ha.extend(preds_split)
    y_true.extend(y_test)
    dates_oos.extend(x_test['Date'])
    tickers_oos.extend(x_test['Ticker'])

    print(f"[Split {split_idx}] R² HA IN-sample: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Conversion en array
y_trainval_true = np.array(y_trainval_true)
y_trainval_pred_ha = np.array(y_trainval_pred_ha)

#général
dates_in = np.array(dates_in)
tickers_in = np.array(tickers_in)
dates_oos = np.array(dates_oos)
tickers_oos = np.array(tickers_oos)
y_true = np.array(y_true)
y_pred_ha = np.array(y_pred_ha)

[Split 1] R² HA IN-sample: 0.019580 | OOS: 0.021224 | SR IN: 0.566 | SR OOS: 0.560
[Split 2] R² HA IN-sample: 0.019650 | OOS: -0.000761 | SR IN: 0.566 | SR OOS: 0.559
[Split 3] R² HA IN-sample: 0.019109 | OOS: 0.018824 | SR IN: 0.565 | SR OOS: 0.597


ALGORITHMES

In [50]:
"""
OLS : Ordinary Least Squares : Nous détaillons ici cet algorithme, la logique étant identique pour les autres modèles.
Listes utilisées : 
- r2_in_sample_list et r2_test_list : stockent, pour chaque split, les R² in‑sample et out‑of‑sample. Elles servent à analyser
  la performance split par split et à ajuster le tuning des modèles (utile pour les modèles à hyperparamètres).
- y_true : valeurs réelles de l’equity premium sur l’ensemble.
- y_pred_ols : prédictions correspondantes du modèle OLS. 
- y_trainval : données d’entraînement (train + validation) utilisées pour l’ajustement du modèle.
- dates_ols et tickers_ols : récupérées à chaque split pour pouvoir fusionner correctement les prédictions de tous les modèles
  et s’assurer que les lignes (dates/tickers) correspondent, évitant tout mélange potentiel des prédictions.

Df et résultats en sortie : 
- df_results_ols : df contenant les prédictions du modèles ols ainsi que la date et le ticker correspondant. 
- r2_results : dictionnaire contenant le r2 ols in sample et oos
"""

#pour calculer les r² globaux 
y_trainval_ols = []

#stocke les r² par split 
r2_in_ols = []
r2_oos_ols = []
y_pred_ols = []

#sucess ratio 
success_ratio_in_ols = []
success_ratio_oos_ols = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    ols = LinearRegression()
    ols.fit(x_trainval, y_trainval)

    #R² in-sample
    y_trainval_pred = ols.predict(x_trainval)
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_in_ols.append(r2_in)

    #R² oos
    y_test_pred = ols.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_oos_ols.append(r2_out)

    #Success ratio
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)
    success_ratio_in_ols.append(sr_in)
    success_ratio_oos_ols.append(sr_out)

    #On stocke tout dans un tableau
    y_pred_ols.append(y_test_pred)
    y_trainval_ols.append(y_trainval_pred) 

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")
    
#On concatène les résultats de tous les splits en un df
y_pred_ols = np.concatenate(y_pred_ols)
y_trainval_ols = np.concatenate(y_trainval_ols)

100%|██████████| 3/3 [00:00<00:00, 46.42it/s]

R² in-sample : 0.038011 | R² oos : 0.014694
R² in-sample : 0.037695 | R² oos : 0.031914
R² in-sample : 0.037578 | R² oos : 0.023721


In [51]:
"""
PLS : Partial Least Squares
Hyperparamètres :
- k : nombre de composantes latentes, choisi pour minimiser la MSE sur la validation.
"""

dates_splits = [] 

# Pour calculer les R² globaux
y_trainval_pls = []

# Stocke les R² par split
r2_in_pls = []
r2_oos_pls = []
y_true_pls = []
y_pred_pls = []

# Success ratio
success_ratio_in_pls = []
success_ratio_oos_pls = []

# Hyperparamètres PLS
best_components_pls = []
mse_val_grids = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    candidate_ks = np.arange(1, 28)
    mse_val_grid = []
    best_mse = float('inf')
    best_k = None

    # Recherche du meilleur k
    for k in candidate_ks:
        pls = PLSRegression(n_components=k, scale=False)
        pls.fit(x_train[covariates], y_train)
        y_val_pred = pls.predict(x_val[covariates]).ravel()
        mse_val = mean_squared_error(y_val, y_val_pred)
        mse_val_grid.append(mse_val)

        if mse_val < best_mse:
            best_mse = mse_val
            best_k = k

    mse_val_grids.append(mse_val_grid)
    best_components_pls.append(best_k)
    
    first_date = x_test['Date'].iloc[0] #récup date split
    dates_splits.append(first_date)

    print(f"Split {split_idx} : meilleur nombre de composantes k = {best_k}")

    # Réentraîner sur train+val avec le meilleur k
    pls_final = PLSRegression(n_components=best_k, scale=False)
    pls_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = pls_final.predict(x_trainval).ravel()
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_pls.append(r2_in)

    # R² oos
    y_test_pred = pls_final.predict(x_test[covariates]).ravel()
    r2_out = r2(y_test, y_test_pred)
    r2_oos_pls.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test.values, y_test_pred)
    success_ratio_in_pls.append(sr_in)
    success_ratio_oos_pls.append(sr_out)

    # Stockage pour global
    y_true_pls.append(y_test)
    y_pred_pls.append(y_test_pred)
    y_trainval_pls.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

y_true_pls = np.concatenate(y_true_pls)
y_pred_pls = np.concatenate(y_pred_pls)

y_trainval_pls = np.concatenate(y_trainval_pls)


 33%|███▎      | 1/3 [00:02<00:05,  2.63s/it]

Split 1 : meilleur nombre de composantes k = 1
R² in-sample : 0.022679 | R² oos : 0.029549


 67%|██████▋   | 2/3 [00:05<00:02,  2.72s/it]

Split 2 : meilleur nombre de composantes k = 4
R² in-sample : 0.031472 | R² oos : 0.021320


100%|██████████| 3/3 [00:08<00:00,  2.79s/it]

Split 3 : meilleur nombre de composantes k = 12
R² in-sample : 0.037575 | R² oos : 0.023761


In [ ]:
from sklearn.pipeline import Pipeline

"""
PCR : Principal Component Regression
Hyperparamètre : k (nombre de composantes principales)
"""

# Pour calculer les R² globaux
y_trainval_pcr = []

# Stocke les R² par split
r2_in_pcr = []
r2_oos_pcr = []
y_true_pcr = []
y_pred_pcr = []

# Success ratio
success_ratio_in_pcr = []
success_ratio_oos_pcr = []

# Hyperparamètres spécifiques PCR
best_components_pcr = []
mse_val_grids_pcr = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    candidate_ks = np.arange(1, 28)
    mse_val_grid = []
    best_mse = float('inf')
    best_k = None

    # Recherche du meilleur k
    for k in candidate_ks:
        pcr_pipe = Pipeline([
            ('pca', PCA(n_components=k)),
            ('reg', LinearRegression())
        ])
        pcr_pipe.fit(x_train[covariates], y_train)
        y_val_pred = pcr_pipe.predict(x_val[covariates]).ravel()
        mse_val = mean_squared_error(y_val, y_val_pred)
        mse_val_grid.append(mse_val)

        if mse_val < best_mse:
            best_mse = mse_val
            best_k = k

    mse_val_grids_pcr.append(mse_val_grid)
    best_components_pcr.append(best_k)
    print(f"Split {split_idx} : meilleur nombre de composantes k = {best_k}")

    # Réentraîner sur train+val avec le meilleur k
    pcr_final = Pipeline([
        ('pca', PCA(n_components=best_k)),
        ('reg', LinearRegression())
    ])
    pcr_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = pcr_final.predict(x_trainval).ravel()
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_pcr.append(r2_in)

    # R² oos
    y_test_pred = pcr_final.predict(x_test[covariates]).ravel()
    r2_out = r2(y_test, y_test_pred)
    r2_oos_pcr.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test.values, y_test_pred)
    success_ratio_in_pcr.append(sr_in)
    success_ratio_oos_pcr.append(sr_out)

    # Stockage pour global
    y_true_pcr.append(y_test)
    y_pred_pcr.append(y_test_pred)
    y_trainval_pcr.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
y_true_pcr = np.concatenate(y_true_pcr)
y_pred_pcr = np.concatenate(y_pred_pcr)
y_trainval_pcr = np.concatenate(y_trainval_pcr)

 33%|███▎      | 1/3 [00:00<00:00,  2.42it/s]

Split 1 : meilleur nombre de composantes k = 4
R² in-sample : 0.019927 | R² oos : 0.025998


 67%|██████▋   | 2/3 [00:00<00:00,  2.35it/s]

Split 2 : meilleur nombre de composantes k = 24
R² in-sample : 0.028911 | R² oos : 0.019573


100%|██████████| 3/3 [00:01<00:00,  2.34it/s]

Split 3 : meilleur nombre de composantes k = 27
R² in-sample : 0.037578 | R² oos : 0.023721


In [53]:
"""
ENet : Elastic Net

Hyperparamètres :
- lambda (alpha) : coefficient de pénalisation choisi pour minimiser la MSE
- l1_ratio fixé à 0.5
"""

# Pour calculer les R² globaux
y_trainval_en = []

# Stocke les R² par split
r2_in_en = []
r2_oos_en = []
y_pred_en = []

# Success ratio
success_ratio_in_en = []
success_ratio_oos_en = []

# Hyperparamètres spécifiques
best_lambdas = []

enet_param_grid = {
    'alpha': np.logspace(-4, 0, num=10)
}

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_lambda = None

    # Recherche du meilleur alpha
    for params in ParameterGrid(enet_param_grid):
        enet = ElasticNet(**params, l1_ratio=0.5, max_iter=10000)
        enet.fit(x_train[covariates], y_train)
        y_val_pred = enet.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        if mse < best_mse:
            best_mse = mse
            best_lambda = params['alpha']

    best_lambdas.append(best_lambda)
    print(f"Split {split_idx} : meilleur lambda = {best_lambda}")

    # Réentraîner sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    en_final = ElasticNet(alpha=best_lambda, l1_ratio=0.5, max_iter=10000)
    en_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = en_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_en.append(r2_in)

    # R² oos
    y_test_pred = en_final.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_oos_en.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)
    success_ratio_in_en.append(sr_in)
    success_ratio_oos_en.append(sr_out)

    # Stockage pour global
    y_pred_en.append(y_test_pred)
    y_trainval_en.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
y_pred_en = np.concatenate(y_pred_en)
y_trainval_en = np.concatenate(y_trainval_en)


 33%|███▎      | 1/3 [00:03<00:06,  3.32s/it]

Split 1 : meilleur lambda = 0.002154434690031882
R² in-sample : 0.022771 | R² oos : 0.030048
Split 2 : meilleur lambda = 0.000774263682681127


 67%|██████▋   | 2/3 [00:11<00:06,  6.04s/it]

R² in-sample : 0.032732 | R² oos : 0.026530
Split 3 : meilleur lambda = 0.0001


100%|██████████| 3/3 [00:18<00:00,  6.17s/it]

R² in-sample : 0.037462 | R² oos : 0.024545


In [70]:
"""
RF : Random Forest
Hyperparamètres :
- n_estimators : nombre d’arbres dans la forêt.
- max_depth : profondeur maximale de chaque arbre.
- min_samples_leaf : nombre minimal d’échantillons dans une feuille.
- max_features : nombre de variables considérées pour le split.
"""

param_grid_rf = {
    'n_estimators': [150, 200, 300],       
    'max_depth': [4,5,6],
    'min_samples_leaf': [3, 5, 10],
    'max_features': ['log2', 2]
}


# Pour calculer les R² globaux
y_trainval_rf = []

# Stocke les R² par split
r2_in_rf = []
r2_oos_rf = []
y_pred_rf = []

# Success ratio
success_ratio_in_rf = []
success_ratio_oos_rf = []

# Hyperparamètres spécifiques
best_params_rf = []
mse_val_grids_rf = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    # Grid search
    for params in ParameterGrid(param_grid_rf):
        rf = RandomForestRegressor(
            **params,
            n_jobs=-1,
            random_state=0
        )
        rf.fit(x_train[covariates], y_train)
        y_val_pred = rf.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_rf.append(mse_grid)
    best_params_rf.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Train sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    rf_final = RandomForestRegressor(
        **best_params,
        n_jobs=-1,
        random_state=0
    )
    rf_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = rf_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_rf.append(r2_in)

    # R² oos
    y_test_pred = rf_final.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_oos_rf.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)
    success_ratio_in_rf.append(sr_in)
    success_ratio_oos_rf.append(sr_out)

    # Stockage pour global
    y_pred_rf.append(y_test_pred)
    y_trainval_rf.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
y_pred_rf = np.concatenate(y_pred_rf)
y_trainval_rf = np.concatenate(y_trainval_rf)


  0%|          | 0/3 [00:00<?, ?it/s]


Split 1 : meilleurs params = {'max_depth': 6, 'max_features': 2, 'min_samples_leaf': 5, 'n_estimators': 150} (MSE val = 0.002993)


 33%|███▎      | 1/3 [00:43<01:26, 43.21s/it]

R² in-sample : 0.070829 | R² oos : 0.037450

Split 2 : meilleurs params = {'max_depth': 6, 'max_features': 'log2', 'min_samples_leaf': 5, 'n_estimators': 150} (MSE val = 0.003457)


 67%|██████▋   | 2/3 [01:34<00:48, 48.08s/it]

R² in-sample : 0.083395 | R² oos : 0.024663

Split 3 : meilleurs params = {'max_depth': 6, 'max_features': 'log2', 'min_samples_leaf': 3, 'n_estimators': 200} (MSE val = 0.006837)


100%|██████████| 3/3 [02:24<00:00, 48.10s/it]

R² in-sample : 0.087657 | R² oos : 0.023019


In [107]:
"""
GBRT : Gradient Boosted Regression Trees
Hyperparamètres :
- n_estimators : nombre d’arbres successifs
- learning_rate : taux d’apprentissage
- max_depth : profondeur maximale des arbres
- loss : fonction de perte
- alpha : paramètre huber
"""
param_grid_gbrt = {
    'n_estimators': [200, 300],       
    'learning_rate': [0.005, 0.01], 
    'max_depth': [2, 3, 4],              
    'loss': ['huber'],
    'alpha': [0.9]
}

# Pour calculer les R² globaux
y_trainval_gbrt = []

# Stocke les R² par split
r2_in_gbrt = []
r2_oos_gbrt = []

y_pred_gbrt = []

# Success ratio
success_ratio_in_gbrt = []
success_ratio_oos_gbrt = []

# Hyperparamètres spécifiques
best_params_gbrt = []
mse_val_grids_gbrt = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    # Grid search
    for params in ParameterGrid(param_grid_gbrt):
        gbrt = GradientBoostingRegressor(**params, random_state=0)
        gbrt.fit(x_train[covariates], y_train)
        y_val_pred = gbrt.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_gbrt.append(mse_grid)
    best_params_gbrt.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Train sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    gbrt_final = GradientBoostingRegressor(**best_params, random_state=0)
    gbrt_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = gbrt_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_gbrt.append(r2_in)

    # R² oos
    y_test_pred = gbrt_final.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_oos_gbrt.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)
    success_ratio_in_gbrt.append(sr_in)
    success_ratio_oos_gbrt.append(sr_out)

    # Stockage pour global
    y_pred_gbrt.append(y_test_pred)
    y_trainval_gbrt.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
y_pred_gbrt = np.concatenate(y_pred_gbrt)
y_trainval_gbrt = np.concatenate(y_trainval_gbrt)


  0%|          | 0/3 [00:00<?, ?it/s]


Split 1 : meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 2, 'n_estimators': 200} (MSE val = 0.003117)


 33%|███▎      | 1/3 [05:18<10:37, 318.58s/it]

R² in-sample : 0.028418 | R² oos : 0.035103

Split 2 : meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 4, 'n_estimators': 300} (MSE val = 0.003463)


 67%|██████▋   | 2/3 [11:48<06:00, 360.75s/it]

R² in-sample : 0.080688 | R² oos : 0.030575

Split 3 : meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 4, 'n_estimators': 300} (MSE val = 0.006812)


100%|██████████| 3/3 [18:48<00:00, 376.31s/it]

R² in-sample : 0.078700 | R² oos : 0.020079


In [ ]:
"""
XGB : XGBoost Regressor
Hyperparamètres :
- n_estimators : nombre d’arbres
- max_depth : profondeur max
- eta : learning rate
"""

param_grid_xgb = {
    'max_depth': [2, 3, 4],
    'learning_rate': [0.01],
    'n_estimators': [300],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0.5, 1, 2]
}

# Pour calculer les R² globaux
y_trainval_xgb = []

# Stocke les R² par split
r2_in_xgb = []
r2_oos_xgb = []
y_true_xgb = []
y_pred_xgb = []

# Pour les portefeuilles
dates_xgb = []
tickers_xgb = []

# Success ratio
success_ratio_in_xgb = []
success_ratio_oos_xgb = []

# Hyperparamètres spécifiques
best_params_xgb = []
mse_val_grids_xgb = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    # Grid search
    for params in ParameterGrid(param_grid_xgb):
        xgb_model = XGBRegressor(**params, random_state=0, n_jobs=-1)
        xgb_model.fit(x_train[covariates], y_train)
        y_val_pred = xgb_model.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_xgb.append(mse_grid)
    best_params_xgb.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Train sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    xgb_final = XGBRegressor(**best_params, random_state=0, n_jobs=-1)
    xgb_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = xgb_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_xgb.append(r2_in)

    # R² oos
    y_test_pred = xgb_final.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_oos_xgb.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)
    success_ratio_in_xgb.append(sr_in)
    success_ratio_oos_xgb.append(sr_out)

    # Stockage pour global
    tickers_xgb.append(x_test["Ticker"])
    dates_xgb.append(x_test["Date"])
    y_pred_xgb.append(y_test_pred)
    y_trainval_xgb.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
dates_xgb = np.concatenate(dates_xgb)
tickers_xgb = np.concatenate(tickers_xgb)
y_pred_xgb = np.concatenate(y_pred_xgb)
y_trainval_xgb = np.concatenate(y_trainval_xgb)

df_results_xgb = pd.DataFrame({
    "Date": dates_xgb,
    "Ticker": tickers_xgb,
    "y_pred_xgb": y_pred_xgb,
})

# Affichages
print(f"\nMoyenne Success ratio in-sample XGB : {np.mean(success_ratio_in_xgb):.6f}")
print(f"Moyenne Success ratio oos XGB : {np.mean(success_ratio_oos_xgb):.6f}")

  0%|          | 0/3 [00:00<?, ?it/s]


Split 1 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 300, 'reg_alpha': 0, 'reg_lambda': 0.5} (MSE val = 0.003035)


 33%|███▎      | 1/3 [00:18<00:37, 18.60s/it]

R² in-sample : 0.063398 | R² oos : 0.055117

Split 2 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 300, 'reg_alpha': 0.1, 'reg_lambda': 1} (MSE val = 0.003423)


 67%|██████▋   | 2/3 [00:38<00:19, 19.26s/it]

R² in-sample : 0.061574 | R² oos : 0.036865

Split 3 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 4, 'n_estimators': 300, 'reg_alpha': 0.5, 'reg_lambda': 0.5} (MSE val = 0.006727)


100%|██████████| 3/3 [00:59<00:00, 19.84s/it]

R² in-sample : 0.094978 | R² oos : 0.021898

Moyenne Success ratio in-sample XGB : 0.581855
Moyenne Success ratio oos XGB : 0.580170


EXPORT

In [108]:
#R2 in-sample oos → moyenne
models = ['ha', 'ols', 'pls', 'pcr', 'en', 'rf', 'gbrt', 'xgb']
rows = []
for model in models:
    r2_in = globals()[f"r2_in_{model}"]
    r2_oos = globals()[f"r2_oos_{model}"]
    row = {
            "Model": model.upper(),
            "Split 1 o": r2_oos[0],
            "Split 2 o": r2_oos[1],
            "Split 3 o": r2_oos[2],
            "Split m o": np.mean(r2_oos),
            "Split 1 i": r2_in[0],
            "Split 2 i": r2_in[1],
            "Split 3 i": r2_in[2],
            "Split m i": np.mean(r2_in),
        }
    rows.append(row)

    df_r2_split = pd.DataFrame(rows)

df_r2_split.to_excel("df_r2_split.xlsx")

In [109]:
#DF → export (comme ça on a pas à refaire tourner model à chaque fois → trop long)

max_depths_rf = [d["max_depth"] for d in best_params_rf]
max_depths_gbrt = [d["max_depth"] for d in best_params_gbrt]
max_depths_xgb = [d["max_depth"] for d in best_params_xgb]

df_best_param = pd.DataFrame({
    "Date": dates_splits,
    "Best_k_pls": best_components_pls,
    "Best_k_pcr": best_components_pcr,
    "Best lambda": best_lambdas,
    "max_depths_rf" : max_depths_rf,
    "max_depths_gbrt" : max_depths_gbrt,
    "max_depths_xgb" : max_depths_xgb
})

df_in = pd.DataFrame({
    "Date": dates_in,
    "Ticker": tickers_in,
    "y_trainval_true": y_trainval_true,
    "y_trainval_pred_ha": y_trainval_pred_ha,
    "y_trainval_ols": y_trainval_ols,
    "y_trainval_pls": y_trainval_pls,
    "y_trainval_pcr": y_trainval_pcr,
    "y_trainval_en": y_trainval_en,
    "y_trainval_rf": y_trainval_rf,
    "y_trainval_gbrt": y_trainval_gbrt,
    "y_trainval_xgb": y_trainval_xgb
})

df_oos = pd.DataFrame({
    "Date": dates_oos,
    "Ticker": tickers_oos,
    "y_true": y_true,
    "y_pred_ols": y_pred_ols,
    "y_pred_pls": y_pred_pls,
    "y_pred_pcr": y_pred_pcr,
    "y_pred_en": y_pred_en,
    "y_pred_rf": y_pred_rf,
    "y_pred_gbrt": y_pred_gbrt,
    "y_pred_xgb": y_pred_xgb,
    "y_pred_ha": y_pred_ha
})

df_best_param.to_excel("df_best_param.xlsx")
df_in.to_excel("df_in.xlsx")
df_oos.to_excel("df_oos.xlsx")

In [110]:
#Calculs métriques out-of-sample : R² moyen, R² par split 

predictions_oos = {
    "OLS" : y_pred_ols,
    "PLS" : y_pred_pls,
    "PCR" : y_pred_pcr,
    "Enet" : y_pred_en,
    "RF" : y_pred_rf,
    "GBRT" : y_pred_gbrt,
    "XGB" : y_pred_xgb,
    "HA" : y_pred_ha
}


rows = []
#récupérer les R²
for model_name, y_pred in predictions_oos.items():
    r2_oos = r2(y_true, y_pred)
    rows.append({
        "Model" : model_name, 
        "Out-of-sample $R^2$": r2_oos,})
df_r2_oos = pd.DataFrame(rows)
print(df_r2_oos)

#vers latex
df_r2_oos_latex = df_r2_oos.copy()
for col in df_r2_oos_latex.columns:
    if col != "Model":  # garde la colonne texte telle quelle
        df_r2_oos_latex[col] = (df_r2_oos_latex[col] * 100).apply(lambda x: f"{x:.2f}")
latex_table = df_r2_oos_latex.to_latex(index=False, escape=False)
print(latex_table)


  Model  Out-of-sample $R^2$
0   OLS             0.024993
1   PLS             0.023963
2   PCR             0.022647
3  Enet             0.026295
4    RF             0.026319
5  GBRT             0.026693
6   XGB             0.033540
7    HA             0.012189
\begin{tabular}{ll}
\toprule
Model & Out-of-sample $R^2$ \\
\midrule
OLS & 2.50 \\
PLS & 2.40 \\
PCR & 2.26 \\
Enet & 2.63 \\
RF & 2.63 \\
GBRT & 2.67 \\
XGB & 3.35 \\
HA & 1.22 \\
\bottomrule
\end{tabular}



PORTEFEUILLES

III. RESULTATS

In [ ]:
df_predict = df_results_ols.merge(
    df_results_pls[['Date', 'Ticker', 'y_pred_pls']], on=['Date', 'Ticker'], how='left'
).merge(
    df_results_pcr[['Date', 'Ticker', 'y_pred_pcr']], on=['Date', 'Ticker'], how='left'
).merge(
    df_results_en[['Date', 'Ticker', 'y_pred_en']], on=['Date', 'Ticker'], how='left'
).merge(
    df_results_rf[['Date', 'Ticker', 'y_pred_rf']], on=['Date', 'Ticker'], how='left'
).merge(
    df_results_gbrt[['Date', 'Ticker', 'y_pred_gbrt']], on=['Date', 'Ticker'], how='left'
).merge(
    df_results_xgb[['Date', 'Ticker', 'y_pred_xgb']], on=['Date', 'Ticker'], how='left'
).merge(
    df_results_ha[['Date', 'Ticker', 'y_true', 'y_pred_ha']], on=['Date', 'Ticker'], how='left'
)

In [ ]:
print(df_predict.head())
cols = ['Date', 'Ticker', 'y_true', 'y_pred_ols', 'y_pred_pls', 'y_pred_pcr', 
        'y_pred_en', 'y_pred_rf', 'y_pred_gbrt', 'y_pred_xgb', 'y_pred_ha']
df_predict = df_predict[cols]
print(df_predict.columns)

In [ ]:
df_me = pd.read_excel("me_df.xlsx")
df_me = df_me.sort_values("Date").reset_index(drop=True) 

In [ ]:
df_all = df_predict.merge(
    df_me[['Date','Ticker','Mkt_Cap_Monthly']],
    on=['Date','Ticker'],
    how='left'
)

In [87]:
# Liste des colonnes de prévisions dans ton DataFrame
pred_cols = ['y_pred_ols','y_pred_pls','y_pred_pcr','y_pred_en',
             'y_pred_rf','y_pred_gbrt','y_pred_xgb','y_pred_ha']

col_ret = 'y_true'
col_weight = 'Mkt_Cap_Monthly'
nb_groups = 3  

all_rows_ew = [] #stocker tous les mois

for date, i in df_all.groupby('Date'):
    i = i.sort_values(col_pred).reset_index(drop=True)  # trie et remet un index clair
    n = len(i) #nombre de ligne par date
    size = n // nb_groups #= taille d'un groupe 
    
    groups = []
    for idx in range(n):
        if idx < size:
            groups.append(0)
        elif idx < 2*size:
            groups.append(1)
        else:
            groups.append(2)
    i["group"]= groups 

    rows_ew = [] #equally weighted 

for grp in sorted(i['group'].unique()):
    sub = i[i['group'] == grp]
    ret_ew_realized = sub[col_ret].mean()
    ret_ew_predicted = sub[col_pred].mean()
    rows_ew.append({
        'Date': date,
        'Group': grp,
        'Return_EW_realized': ret_ew_realized,
        'Return_EW_predicted': ret_ew_predicted
    })


df_ew = pd.DataFrame(all_rows_ew)
pivot_ew = df_ew.pivot(index='Date', columns='Group', values='Return_EW_realized')

pivot_ew['HighLow'] = pivot_ew[nb_groups-1] - pivot_ew[0]


mean_hl = pivot_ew['HighLow'].mean()
std_hl = pivot_ew['HighLow'].std()
sharpe_hl = (mean_hl / std_hl) * np.sqrt(12)
t_stat = mean_hl / (std_hl/np.sqrt(len(pivot_ew)))

print("Equally weighted results")
print(f"Mean H-L: {mean_hl:.4f}")
print(f"Sharpe H-L: {sharpe_hl:.4f}")
print(f"t-stat: {t_stat:.2f}")


NameError: name 'df_all' is not defined